In [1]:
# Cell 1 – FIRST, before any app imports
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# Explicit path; Jupyter project root is usually /app or /code
for p in [Path("/app/.env"), Path("/code/.env"), Path.cwd() / ".env"]:
    if p.exists():
        load_dotenv(p)
        print(f"Loaded from {p}")
        break
else:
    print("No .env found")

# Sanity check
print("HUGGINGFACE_API_TOKEN set:", bool(os.getenv("HUGGINGFACE_API_TOKEN")))
print("OPENAI_API_TOKEN set:", bool(os.getenv("OPENAI_API_TOKEN")))
# override DATABASE_URL to look at local
os.environ["DATABASE_URL"] = "postgresql+psycopg2://badger:badgerpass@db:5432/badgerdb"
sys.path.append(str(Path().resolve().parent))
from app.db import engine, SessionLocal#from app.models import User, Document, VaultMembership
from app.topics import recluster_vaults_with_tree #as rct
from uuid import UUID

Loaded from /app/.env
HUGGINGFACE_API_TOKEN set: True
OPENAI_API_TOKEN set: False


In [2]:
import numpy as np
%pip install pandas
#!{sys.executable} -m pip install pandas

import pandas as pd
import json
#!pip install ipywidgets
#! pip install matplotlib



Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
# test@example.com vault.id '756a8b1d-e92f-4ce0-a4bf-b8bc4ec2e1b7'
# user.id '2593557a-8849-41b6-b3e3-c1420ddd0c6a'

db = SessionLocal()
try:
    user_id = UUID("2593557a-8849-41b6-b3e3-c1420ddd0c6a")  # Your user's UUID
    vault_ids = [UUID("756a8b1d-e92f-4ce0-a4bf-b8bc4ec2e1b7")]  # Specific vault(s) to recluster
    result = recluster_vaults_with_tree(
        db, user_id=user_id, days=365,
        vault_ids=vault_ids,
        relabel=True,
    )
    print(result)
finally:
    db.close()

RuntimeError: HuggingFace embed HTTP 400: Bad Request. Body: {"error":"Bad Request: Your endpoint is in error, check its status on endpoints.huggingface.co","code":"BAD_REQUEST"}

In [4]:
from app.helpers import get_openai_client

In [5]:
openai_models = ["gpt-4.1-nano", "gpt-4.1-mini", "gpt-4.1", "gpt-4o-mini", "gpt-4o"]
model_name = "gpt-4.1-mini"

# Initialize client
client = get_openai_client()



In [6]:
def generate_doc_prompt(doc):
    doc_payload_prompt = (
    "You are creating a descriptive payload from this document for the purposes of creating a topic label."
    "The payload should provide a sampling of multiple (3) samples for each of the indicated categories below.\n"
    "We want to capture multiple aspects of the document.  Sample from multiple portions of the distribution.\n"
    "Return only valid JSON with double quotes. No markdown. No explanation.\n"
    "document type should not descibe the subject of the document, it should describe the format like question, prompt, note\n"
    "purpose or intent should not desribe subject matter.  it should descibe why the author is writing the note - to convice, to ask, to describe\n"
    "the discriminative subtitle should help differentiate the docuement from documents that have similar canonical label,short label or keyphrases.\n"
    "I would like the payload to look something like this:\n"
    """
    {'canonical label': ['description of contrastic topic labeling','alternative ways to describe document clusters']
    ,'short label':['topic modeling', 'cluster labeling', 'contrastic labeling']
    ,'discriminative subtitle':['non-typical labeling','extractive NLP techniques']
    ,'top phrases':['decision boundaries in a tree','isolate topics, usage, ideas that semantically link a branch and descendants']
    ,'purpose or intent':['description of process','educational','convincing','to do list'] #keep this general
    ,'target audience':['data scientists','LLM users','LNP modelers', 'students','professionals','lawyeres','traders']
    ,'document type':['short note','LLM response','query','article','advertisement','short note']
    , 'named entities':['John Bon Jovi','Microsoft','Chicago'}
    """
    f"document:\n{doc}\n"
    "Document payload:\n"
    )
    return doc_payload_prompt


In [7]:
def generate_doc_payload(prompt,model_name="gpt-4.1-mini"):
    #doc = doc_a
    #client = get_openai_client()

    resp = get_openai_client().chat.completions.create(
            model=model_name,
            response_format={"type": "json_object"},
            messages=[{"role": "user", "content": prompt}],
        )
    resp
    s = resp.choices[0].message.content.strip()
    payload = json.loads(s)
    return payload

In [57]:
payload_a = generate_doc_payload(generate_doc_prompt(doc_a),model_name="gpt-4.1")

In [58]:
payload_b = generate_doc_payload(generate_doc_prompt(doc_a),model_name="gpt-4.1")

In [55]:
doc_b_payload

{'canonical label': ['labeling techniques for agglomerative clustering trees',
  'efficient asynchronous cluster topic labeling',
  'contrasting topics within hierarchical document clusters'],
 'short label': ['cluster labeling',
  'tree-based topic modeling',
  'contrastic labeling'],
 'discriminative subtitle': ['minimizing LLM dependency in hierarchical labeling',
  'labeling branches with semantically distinctive topics',
  'dynamic Steiner tree cuts and taxonomic similarity assignment'],
 'top phrases': ['labeling for agglomerative clustering',
  'isolate topics, usage, ideas that semantically link a branch and its descendent',
  'integrate similarity with taxonomies or ontologies'],
 'purpose or intent': ['exploratory analysis',
  'to describe approaches',
  'comparison of methods'],
 'target audience': ['data scientists', 'LLM users', 'NLP modelers'],
 'document type': ['short note', 'exploratory prompt', 'technical query']}

In [8]:
def generate_sibling_prompt(payload_a,payload_b):
    prompt = (
        "You are describing the differences and similarities between the desciptions of two related sets of documents.\n"
        "these will be sibling nodes in an aglomerative tree.  They likely will have many similarities.\n"
        "You want to generate multiple (3) labels that are human interpetable that clearly separate the two sets of documents.\n"
        "You will also generate multiple short tags (3-4 words) that quickly differentiate the sets.\n"
        "You will get two payloads in json form.  Use the payloads to create a differentiation in this format:\n"
        """{"sibling_a":{"contrast":["label 1","label 2","label3"],"contrast_tag":["a","b","c"]}
        ,"sibling_b":{"contrast":["label 1","label 2","label3"],"contrast_tag":["a","b","c"]}"""
        "You will also generate a combined payload for the parent node that combines the two payloads.\n"
        """It should be appended after the sibling payloads in this format: "parent_payload":\n"""
        "The parent payload response should have the same format and categories of as the sibling payloads.\n"
        "the parent payload should highlight the similarities between the two nodes in human readable form.\n"
        "generate multiple labels for each section that capture the breadth of the sibling nodes.\n"
        "Return only valid JSON with double quotes. No markdown. No explanation.\n\n"
        f"payload_a: {payload_a}\n"
        f"payload_b: {payload_b}\n"
        "response:\n"
        
    )
    return prompt

In [9]:
def generate_parent_prompt(payload_a,payload_b):
    prompt = (
        "You are describing the differences and similarities between the desciptions of two related sets of documents.\n"
        "these will be sibling nodes in an aglomerative tree.  They likely will have many similarities.\n"
        "You will get two payloads in json form.\n"
        "You will generate a combined payload for the parent node that combines the two payloads.\n"
        "The parent payload response should have the same format and categories of as the sibling payloads.\n"
        "the parent payload should highlight the similarities between the two nodes in human readable form.\n"
        "generate multiple labels for each section that capture the breadth of the sibling nodes.\n"
        "Return only valid JSON with double quotes. No markdown. No explanation.\n\n"
        f"payload_a: {payload_a}\n"
        f"payload_b: {payload_b}\n"
        "response:\n"
        
    )
    return prompt

In [10]:
def generate_parent(payload_a,payload_b):
    prompt = generate_parent_prompt(payload_a,payload_b)
    parent_payload = generate_doc_payload(prompt,model_name="gpt-4.1")
    return parent_payload

In [11]:
def generate_parent_sibs(payload_a,payload_b):
    prompt = generate_sibling_prompt(payload_a,payload_b)
    sib_payload = generate_doc_payload(prompt,model_name="gpt-4.1")
    parent_payload = sib_payload['parent_payload']
    payload_a.update(sib_payload['sibling_a'])
    payload_b.update(sib_payload['sibling_b'])
    return payload_a, payload_b, parent_payload

In [80]:
sib_payload

{'sibling_a': {'contrast': ['Contrastive and discriminative labeling focus',
   'Long-form structured recommendations and design principles',
   'Broader audience including topic modelers and product developers'],
  'contrast_tag': ['contrastive labeling methods',
   'comprehensive best practices',
   'multi-disciplinary audience']},
 'sibling_b': {'contrast': ['Emphasis on overcoming LLM/tf-idf limitations in cluster labeling',
   'Concise notes focusing on solutions, problem statements, and advice',
   'Targeted toward technical cluster labeling in agglomerative hierarchies'],
  'contrast_tag': ['llm/tfidf alternatives',
   'solution-focused notes',
   'technical hierarchical labeling']},
 'parent_payload': {'contrast': ['Advanced labeling strategies for hierarchical document clustering',
   'Shared challenge of discriminative labeling for sibling and branch nodes',
   'Emphasis on semantic separation within tree-based structures'],
  'contrast_tag': ['hierarchical label strategies',

In [84]:
#payload_a['sibling contrast'] = sib_payload['sibling_a']
#payload_b['sibling contrast'] = sib_payload['sibling_b']
payload_a.update(sib_payload['sibling_a'])
payload_b.update(sib_payload['sibling_b'])

parent_payload = sib_payload['parent_payload']

In [12]:
def generate_refinement_prompt(parent_payload,child_payload):
    prompt = (
        "You are describing the differences between the desciptions of a parent node and a child node.\n"
        "these will be nodes in an aglomerative tree.  They likely will have many similarities.\n"
        "You want to generate multiple (3) labels that are human interpetable that clearly separate the two payloads.\n"
        "The labels for the children should refine the labels supplied by the parent like:\n"
        "parent: customer support\n"
        "child: refund disputes\n"
        "instead of repeating customer/support/service everywhere.\n\n"
        "You will also generate multiple short tags (3-4 words) that quickly differentiate the sets.\n"
        "You will get two payloads in json form.  Use the payloads to create a differentiation in this format:\n"
        """{{"parent_refine":["label 1","label 2","label3"],"parent_refine_tag":["a","b","c"]}"""
        "Return only valid JSON with double quotes. No markdown. No explanation.\n\n"
        f"parent_payload: {parent_payload}\n"
        f"child_payload: {child_payload}\n"
        "response:\n"
        
    )
    return prompt

In [13]:
def refine_child(parent_payload,child_payload):
    prompt = generate_refinement_prompt(parent_payload,child_payload)
    refine_payload = generate_doc_payload(prompt,model_name="gpt-4.1")
    child_payload.update(refine_payload)
    return child_payload

In [86]:
payload_a

{'canonical label': ['contrastive clustering tree labeling approaches',
  'discriminative topic label generation in hierarchies',
  'relative labeling for hierarchical document trees'],
 'short label': ['contrastive labeling',
  'hierarchical topic labels',
  'discriminative branch labels'],
 'discriminative subtitle': ['methods for distinguishing sibling nodes in tree-based clusters',
  'multi-axis and taxonomy-aided label generation',
  'contrastive strategies for non-redundant internal node labeling'],
 'top phrases': ['decision boundaries in a tree',
  'isolate topics, usage, ideas that semantically link a branch and descendants',
  'node-vs-sibling contrastive labeling',
  'sibling distinguishability and label non-redundancy',
  'multi-scale, exemplar-based cluster labeling'],
 'purpose or intent': ['description of process',
  'outline of design principles',
  'recommendation of best practices'],
 'target audience': ['data scientists',
  'topic modelers',
  'machine learning pract

In [14]:
qry = """select document_id , chunk_index, chunk_text
from embedding.all_minilm_v1_384 amv
where document_id ='e628d6af-ce66-4136-b245-382d9706724b'
"""

df = pd.read_sql(qry, engine)


In [15]:
df.tail()
#generate_doc_payload(generate_doc_prompt(doc_a),model_name="gpt-4.1-mini")

,document_id,chunk_index,chunk_text
16,e628d6af-ce66-4136-b245-382d9706724b,0,Get 10k free credits when you sign up for Llam...
17,e628d6af-ce66-4136-b245-382d9706724b,7,Execute hyperparameter search\nresults = param...
18,e628d6af-ce66-4136-b245-382d9706724b,8,LlamaIndex Recursive Retrieval Recipe (noteboo...
19,e628d6af-ce66-4136-b245-382d9706724b,10,"response = query_engine_chunk.query(\n ""Can..."
20,e628d6af-ce66-4136-b245-382d9706724b,17,flare_query_engine = FLAREInstructQueryEngine(...


In [16]:
ct = df['chunk_text'].tolist()
ct[0]

'The RAG cheat sheet shared above was greatly inspired by a recent RAG survey paper (“Retrieval-Augmented Generation for Large Language Models: A Survey” Gao, Yunfan, et al. 2023). Basic RAG\n\nMainstream RAG as defined today involves retrieving documents from an external knowledge database and passing these along with the user’s query to an LLM for response generation. In other words, RAG involves a Retrieval component, an External Knowledge database and a Generation component. LlamaIndex Basic RAG Recipe:\n\nfrom llama_index import SimpleDirectoryReader, VectorStoreIndex\n\n# load data\ndocuments = SimpleDirectoryReader(input_dir="...").load_data()\n\n# build VectorStoreIndex that takes care of chunking documents\n# and encoding chunks to embeddings for future retrieval\nindex = VectorStoreIndex.from_documents(documents=documents)\n\n#'

In [36]:
payloads = [generate_doc_payload(generate_doc_prompt(chunk),model_name="gpt-4.1-mini") for chunk in ct]
#payloads_copy = payloads

In [17]:
from math import log2
from typing import Any, Callable, List


Payload = Any


def largest_power_of_two_leq(n: int) -> int:
    """Return the largest power of 2 less than or equal to n."""
    if n < 1:
        raise ValueError("n must be >= 1")
    return 1 << (n.bit_length() - 1)


def initial_play_in_round(
    payloads: List[Payload],
    generate_parent: Callable[[Payload, Payload], Payload],
) -> List[Payload]:
    """
    Reduce the list size down to the nearest lower power of 2 by merging
    the first 2*m payloads pairwise, where m = n - largest_power_of_two_leq(n).
    """
    n = len(payloads)
    if n <= 1:
        return payloads[:]

    target = largest_power_of_two_leq(n)
    m = n - target

    # Merge the first 2*m payloads pairwise
    merged = []
    for i in range(0, 2 * m, 2):
        parent = generate_parent(payloads[i], payloads[i + 1])
        merged.append(parent)

    # Keep the rest untouched
    merged.extend(payloads[2 * m :])
    return merged


def tournament_reduce(
    payloads: List[Payload],
    generate_parent: Callable[[Payload, Payload], Payload],
    verbose: bool = False,
) -> Payload:
    """
    Reduce an ordered list of payloads using:
      1. a play-in round to reach a power of 2
      2. repeated adjacent pairwise merges until one payload remains
    """
    if not payloads:
        raise ValueError("payloads must not be empty")

    if len(payloads) == 1:
        return payloads[0]

    current = payloads[:]

    if verbose:
        print(f"Start: {len(current)} payloads")

    # Play-in round
    if len(current) & (len(current) - 1) != 0:  # not already a power of 2
        current = initial_play_in_round(current, generate_parent)
        if verbose:
            print(f"After play-in round: {len(current)} payloads")

    # Main tournament rounds
    round_num = 1
    while len(current) > 1:
        if len(current) % 2 != 0:
            raise RuntimeError(
                f"Expected even number of payloads during tournament, got {len(current)}"
            )

        next_round = []
        for i in range(0, len(current), 2):
            parent = generate_parent(current[i], current[i + 1])
            next_round.append(parent)

        current = next_round

        if verbose:
            print(f"After round {round_num}: {len(current)} payloads")

        round_num += 1

    return current[0]

In [52]:
#def generate_parent(a, b):
#    return f"({a}+{b})"


#payloads = ["p1", "p2", "p3", "p4", "p5"]

final_payload = tournament_reduce(payloads, generate_parent, verbose=True)
print(final_payload)

Start: 21 payloads
After play-in round: 16 payloads
After round 1: 8 payloads
After round 2: 4 payloads
After round 3: 2 payloads
After round 4: 1 payloads
{'canonical label': ['advanced retrieval-augmented generation (RAG) system pipelines, evaluation frameworks, and AI product methodologies', 'hybrid LLM-enabled retrieval, evaluation, postprocessing, chunking, and generation architectures', 'integrated workflow and assessment strategies: optimization, document structuring, reranking, context management, and multi-level evaluation', 'synergistic ML/NLP workflows for robust retrieval, generation, and output alignment', 'end-to-end RAG and AI system construction, tuning, assessment, and practical solution development'], 'short label': ['RAG pipeline optimization, evaluation & LLM workflow integration', 'advanced retrieval, chunking, reranking, and generation orchestration', 'multi-step reasoning, iterative retrieval, and workflow enhancement', 'context-aware retrieval-generation synergy

In [18]:
import json


def chunk_list(items, batch_size):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]


def generate_doc_batch_prompt(docs):
    doc_blocks = [f"DOCUMENT {i}:\n{doc}" for i, doc in enumerate(docs)]
    docs_text = "\n\n".join(doc_blocks)

    return f"""
You are creating descriptive payloads from documents for topic labeling.

Return only valid JSON with this exact top-level structure:
{{
  "documents": [
    {{
      "canonical label": ["...", "...", "..."],
      "short label": ["...", "...", "..."],
      "discriminative subtitle": ["...", "...", "..."],
      "top phrases": ["...", "...", "..."],
      "purpose or intent": ["...", "...", "..."],
      "target audience": ["...", "...", "..."],
      "document type": ["...", "...", "..."],
      "named entities": ["...", "...", "..."]
    }}
  ]
}}

Rules:
- Return one payload per document, in the same order as input
- Exactly 3 items per field
- JSON only
- No markdown
- No explanation
- "document type" describes format, not subject
- "purpose or intent" describes author intent, not subject
- "discriminative subtitle" should distinguish from nearby related documents

Documents:
{docs_text}
"""


def generate_doc_payload_batch(docs, model_name="gpt-4.1-mini"):
    client = get_openai_client()
    prompt = generate_doc_batch_prompt(docs)

    resp = client.chat.completions.create(
        model=model_name,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}],
    )

    s = resp.choices[0].message.content.strip()
    parsed = json.loads(s)
    return parsed["documents"]


def generate_all_doc_payloads(docs, batch_size=6, model_name="gpt-4.1-mini"):
    all_payloads = []

    for batch in chunk_list(docs, batch_size):
        payloads = generate_doc_payload_batch(batch, model_name=model_name)
        all_payloads.extend(payloads)

    return all_payloads

In [55]:
chunk_payloads = generate_all_doc_payloads(ct, batch_size=6, model_name="gpt-4.1")

In [60]:
chunk_payloads[0]

{'canonical label': ['Retrieval-Augmented Generation Basics',
  'External Knowledge Integration in LLMs',
  'LlamaIndex Implementation Overview'],
 'short label': ['Basic RAG', 'RAG Components', 'Getting Started with RAG'],
 'discriminative subtitle': ['Primer on standard RAG setup with LlamaIndex',
  'Introduction to document retrieval for LLMs',
  'Outline of RAG system dependencies'],
 'top phrases': ['retrieving documents from knowledge database',
  'llama_index Basic RAG Recipe',
  'retrieval and generation component'],
 'purpose or intent': ['Summarize foundational RAG concepts',
  'Introduce LlamaIndex for RAG workflows',
  'Share a concise RAG implementation example'],
 'target audience': ['LLM practitioners',
  'ML engineers',
  'Developers new to RAG'],
 'document type': ['cheat sheet',
  'tutorial excerpt',
  'implementation walkthrough'],
 'named entities': ['LlamaIndex', 'Gao, Yunfan', 'RAG survey paper']}

In [57]:
payloads[0]

{'canonical label': ['retrieval augmented generation techniques',
  'RAG workflows and architecture',
  'integration of external knowledge with LLMs'],
 'short label': ['RAG overview',
  'index building',
  'retrieval generation pipeline'],
 'discriminative subtitle': ['basic RAG components explained',
  'example with LlamaIndex implementation',
  'summary inspired by recent RAG survey paper'],
 'top phrases': ['retrieving documents from external knowledge database',
  'passing user queries to LLM for response generation',
  'VectorStoreIndex for chunking and embedding documents'],
 'purpose or intent': ['to describe a method',
  'educational',
  'informative overview'],
 'target audience': ['data scientists',
  'machine learning engineers',
  'NLP researchers'],
 'document type': ['short note',
  'technical summary',
  'code snippet explanation'],
 'named entities': ['Gao, Yunfan', 'LlamaIndex', 'VectorStoreIndex']}

In [19]:
def chunk_list(items, batch_size):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]

import json
def make_adjacent_pairs(payloads):
    if len(payloads) % 2 != 0:
        raise ValueError("Number of payloads must be even to make adjacent pairs.")
    return [(payloads[i], payloads[i + 1]) for i in range(0, len(payloads), 2)]

def generate_parent_batch_prompt(payload_pairs):
    """
    payload_pairs: list[tuple[dict, dict]]
    Returns a single prompt asking for one parent payload per sibling pair.
    """
    blocks = []

    for i, (payload_a, payload_b) in enumerate(payload_pairs):
        a_json = json.dumps(payload_a, ensure_ascii=False)
        b_json = json.dumps(payload_b, ensure_ascii=False)

        blocks.append(
            f"PAIR {i}\n"
            f"payload_a:\n{a_json}\n\n"
            f"payload_b:\n{b_json}"
        )

    pairs_text = "\n\n".join(blocks)

    prompt = f"""
You are describing the differences and similarities between two related sets of documents.

Each pair represents sibling nodes in an agglomerative tree.
Sibling nodes will often have substantial overlap, but may differ in emphasis, specificity, audience, format, or intent.

For EACH pair:
- Generate a combined payload for the parent node
- The parent payload must use the same categories as the sibling payloads
- The parent payload should emphasize shared themes while still preserving the breadth of both children
- The parent payload should remain human-readable
- Generate multiple candidate labels for each section
- The parent should be broader than either child, but still discriminative

Return only valid JSON with this exact top-level structure:
{{
  "parents": [
    {{
      "canonical label": ["...", "...", "..."],
      "short label": ["...", "...", "..."],
      "discriminative subtitle": ["...", "...", "..."],
      "top phrases": ["...", "...", "..."],
      "purpose or intent": ["...", "...", "..."],
      "target audience": ["...", "...", "..."],
      "document type": ["...", "...", "..."],
      "named entities": ["...", "...", "..."]
    }}
  ]
}}

Rules:
- Return one parent payload per input pair, in the same order
- Exactly 3 items per field
- JSON only
- No markdown
- No explanation
- Preserve breadth from both children
- "document type" should describe format, not topic
- "purpose or intent" should describe why the author is writing
- "discriminative subtitle" should distinguish this node from nearby related nodes at similar levels of abstraction

Sibling pairs:
{pairs_text}
"""
    return prompt
    
def generate_parent_payload_batch(payload_pairs, model_name="gpt-4.1-mini"):
    """
    payload_pairs: list[tuple[dict, dict]]
    returns: list[dict] parent payloads in same order
    """
    client = get_openai_client()
    prompt = generate_parent_batch_prompt(payload_pairs)

    resp = client.chat.completions.create(
        model=model_name,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}],
    )

    s = resp.choices[0].message.content.strip()
    parsed = json.loads(s)
    return parsed["parents"]

def generate_parent_payload_batch_chunked(payload_pairs, batch_size=8, model_name="gpt-4.1-mini"):
    all_parents = []
    for batch in chunk_list(payload_pairs, batch_size):
        parents = generate_parent_payload_batch(batch, model_name=model_name)
        all_parents.extend(parents)
    return all_parents

def initial_play_in_round_batched(payloads, model_name="gpt-4.1-mini", parent_batch_size=8):
    n = len(payloads)
    if n <= 1:
        return payloads[:]

    target = largest_power_of_two_leq(n)
    m = n - target

    play_in_pairs = [(payloads[i], payloads[i + 1]) for i in range(0, 2 * m, 2)]
    merged = generate_parent_payload_batch_chunked(
        play_in_pairs,
        batch_size=parent_batch_size,
        model_name=model_name,
    )

    merged.extend(payloads[2 * m :])
    return merged


def run_parent_round(payloads, model_name="gpt-4.1-mini", parent_batch_size=8):
    pairs = make_adjacent_pairs(payloads)
    return generate_parent_payload_batch_chunked(
        pairs,
        batch_size=parent_batch_size,
        model_name=model_name,
    )


def tournament_reduce_batched(payloads, model_name="gpt-4.1-mini", parent_batch_size=8, verbose=False):
    if not payloads:
        raise ValueError("payloads must not be empty")

    if len(payloads) == 1:
        return payloads[0]

    current = payloads[:]

    if verbose:
        print(f"Start: {len(current)} payloads")

    if len(current) & (len(current) - 1) != 0:
        current = initial_play_in_round_batched(
            current,
            model_name=model_name,
            parent_batch_size=parent_batch_size,
        )
        if verbose:
            print(f"After play-in round: {len(current)} payloads")

    round_num = 1
    while len(current) > 1:
        current = run_parent_round(
            current,
            model_name=model_name,
            parent_batch_size=parent_batch_size,
        )
        if verbose:
            print(f"After round {round_num}: {len(current)} payloads")
        round_num += 1

    return current[0]

In [70]:
final_payload = tournament_reduce_batched(chunk_payloads, model_name="gpt-4.1", verbose=True)

Start: 21 payloads
After play-in round: 16 payloads
After round 1: 8 payloads
After round 2: 4 payloads
After round 3: 2 payloads
After round 4: 1 payloads


In [71]:
final_payload

{'canonical label': ['Unified RAG Pipeline Design, Evaluation Methods, and Ecosystem Integration',
  'Comprehensive Guide to RAG Workflow Optimization, Quality Assurance, and Industry Practices',
  'Integrated Approaches to Retrieval-Augmented Generation: Pipeline Building, Assessment, and Product Adoption'],
 'short label': ['End-to-End RAG Systems & Ecosystem Resources',
  'Holistic Workflow Design & Assessment',
  'Advanced RAG Pipelines & Industry Integration'],
 'discriminative subtitle': ['Spanning implementation, evaluation, resource integration, and best practices for robust RAG pipelines',
  'Merging technical design, retrieval tuning, safety, and product adoption frameworks for RAG systems',
  'Comprehensive synthesis of pipeline construction, ecosystem tools, and evaluation strategies'],
 'top phrases': ['retrieval and generation optimization, relevancy metrics, and workflow best practices',
  'pipeline building, re-ranking, chunk sizing, and safety-focused strategies',
  'L

# Langchain process

In [21]:
#! pip install -U langchain
#! pip install -U langchain-openai
#! pip install -U langchain_text_splitters

import asyncio
from typing import Any, Optional, List

from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI


class KnowledgePayload(BaseModel):
    title: Optional[str] = None
    summary: str
    key_topics: list[str] = Field(default_factory=list)
    entities: list[str] = Field(default_factory=list)
    facts: list[str] = Field(default_factory=list)
    intent_purpose: list[str] = Field(default_factory=list)
    target_audience: list[str] = Field(default_factory=list)
    document_type: list[str] = Field(default_factory=list)
    industry: list[str] = Field(default_factory=list)
    life_domain: list[str] = Field(default_factory=list)
    provenance: list[str] = Field(default_factory=list)

class SiblingContrast(BaseModel):
    contrast: List[str] = Field(default_factory=list)
    contrast_tag: List[str] = Field(default_factory=list)


class ParentRefinement(BaseModel):
    parent_refine: List[str] = Field(default_factory=list)
    parent_refine_tag: List[str] = Field(default_factory=list)


class BranchEnrichment(BaseModel):
    sibling_a: SiblingContrast
    sibling_b: SiblingContrast
    parent_payload: KnowledgePayload
    parent_refinement_for_a: ParentRefinement
    parent_refinement_for_b: ParentRefinement

extract_prompt = ChatPromptTemplate.from_template("""
You are creating a structured knowledge-base payload for a document.

Return a JSON object matching the schema exactly.

Document metadata:
- document_id: {document_id}
- source: {source}
- title: {title}

Instructions:
- Extract the most important knowledge from the document.
- Be concise but complete.
- Normalize duplicate ideas.
- Prefer concrete facts over vague themes.
- "document_type" describes format, not subject
- "intent_purpose" describes author intent, not subject (convince, educate, imperitive, advertise)
- "industry" describes something like a GICS industry group or industry
- "life_domain" describes where a person would apply this: career, relationships, home, healthcare, vacation, finances, worship,education, politics, legal 

Document text:
{document_text}
""")

reduce_prompt = ChatPromptTemplate.from_template("""
You are consolidating chunk-level knowledge-base payloads into one final payload.

Return a JSON object matching the schema exactly.

Document metadata:
- document_id: {document_id}
- source: {source}
- title: {title}

Chunk payloads:
{chunk_payloads}
""")

#extract_chain_prompt = ChatPromptTemplate.from_template(extract_prompt)
#reduce_chain_prompt = ChatPromptTemplate.from_template(reduce_prompt)


def build_model(model_name: str = "gpt-4.1-mini"):
    llm = ChatOpenAI(model=model_name, temperature=0)
    return llm.with_structured_output(KnowledgePayload, method="json_schema")


splitter = RecursiveCharacterTextSplitter(
    chunk_size=50000,
    chunk_overlap=1000,
    separators=["\n\n", "\n", ". ", " ", ""],
)

async def build_doc_payload(doc: dict[str, Any], model_name="gpt-4.1-mini"):
    model = build_model(model_name)
    extract_chain = extract_prompt | model
    reduce_chain = reduce_prompt | model

    try:
        text = doc["text"] or ""
        chunks = splitter.split_text(text) if len(text) > 180000 else [text]

        if len(chunks) == 1:
            result = await extract_chain.ainvoke({
                "document_id": doc["id"],
                "title": doc.get("title", ""),
                "source": doc.get("source", ""),
                "document_text": chunks[0],
            })
            return {
                "document_id": doc["id"],
                "status": "ok",
                "payload": result.model_dump(),
            }

        chunk_results = await asyncio.gather(*[
            extract_chain.ainvoke({
                "document_id": doc["id"],
                "source": doc.get("source", ""),
                "title": doc.get("title", ""),
                "document_text": chunk,
            })
            for chunk in chunks
        ], return_exceptions=True)

        good_chunks = [r.model_dump() for r in chunk_results if not isinstance(r, Exception)]

        if not good_chunks:
            return {
                "document_id": doc["id"],
                "status": "error",
                "error": "all_chunks_failed",
            }

        reduced = await reduce_chain.ainvoke({
            "document_id": doc["id"],
            "source": doc.get("source", ""),
            "title": doc.get("title", ""),
            "chunk_payloads": good_chunks,
        })

        return {
            "document_id": doc["id"],
            "status": "ok",
            "payload": reduced.model_dump(),
            "chunk_count": len(chunks),
            "successful_chunks": len(good_chunks),
        }

    except Exception as e:
        return {
            "document_id": doc["id"],
            "status": "error",
            "error": str(e),
        }

async def process_documents(docs, model_name="gpt-4.1-mini", max_concurrency=8):
    sem = asyncio.Semaphore(max_concurrency)

    async def run_one(doc):
        async with sem:
            return await build_doc_payload(doc, model_name=model_name)

    return await asyncio.gather(*[run_one(doc) for doc in docs], return_exceptions=False)

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 477.1 kB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 834.4 kB/s eta 0:00:000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 606.0 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 441.2 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.7/51.7 kB 497.2 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.7/96.7 kB 388.6 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.9/385.9 kB 474.4 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.7/345.7 kB 554.4 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.9/193.9 kB 691.7 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.6/133.6 kB 900.5 kB/s eta 0:00:000:010:01
   ━━━━━━━━━━━━━━━━━━━

In [22]:
qry = """select id, url as source, title, full_text as text
from public.documents d
where exists (
	select doc_id from graph.v_tree_map tm
	where tree_id = '75e5abf9-bf3a-421c-872a-e3a5b81d2eec'
	and leaf = true
	and d.id=tm.doc_id)
"""



df = pd.read_sql(qry, engine)
docs = df.to_dict(orient="records")


In [23]:
docs = df.iloc[70:].to_dict(orient="records")
#for i,ddd in enumerate(docs):
#    print(i,ddd['id'])#['47d74543-d474-4498-9edf-ec61726a80dd']
#docs = [docs[9]]

In [24]:
#pay_dict= {}
results = await process_documents(docs=docs, model_name="gpt-4.1-mini")

errors = [r for r in results if r["status"] != "ok"]

pay_dict.update({
    r["document_id"]: r["payload"]
    for r in results
    if r["status"] == "ok"
})

CancelledError: 

In [122]:
len(errors)
errors

[]

In [130]:
len(pay_dict.keys())
df_pay_dict = pd.DataFrame(pay_dict).T.reset_index()
df_pay_dict.rename(columns={'index':'doc_id'},inplace=True)
df_pay_dict.head()
df_pay_dict.to_csv('df_pay_dict.csv',index=False)

In [25]:
branch_prompt = ChatPromptTemplate.from_template("""
Create a combined knowledge payload for a parent branch from two child payloads.

Return JSON matching the schema exactly.

Parent branch_id: {branch_id}
Left child node_id: {left_node_id}
Right child node_id: {right_node_id}

Left payload:
{left_payload}

Right payload:
{right_payload}

Instructions:
- Merge and synthesize, do not just concatenate.
- Preserve the most important facts, entities, topics, audience, intent, and life-domain signals.
- Add provenance entries referencing both child node IDs.
""")

In [69]:
branch_template = ChatPromptTemplate.from_template(
    """
You are analyzing two sibling nodes in a hierarchical knowledge tree.

Each sibling node has a JSON knowledge payload with:
title, summary, key_topics, entities, facts, intent_purpose, target_audience,
document_type, industry, life_domain, provenance.

Task:

1) Sibling contrast
- For each sibling, generate 3 human-interpretable labels that highlight how it
  differs from its sibling.
- Generate 3 short tags (3–4 words) per sibling that quickly distinguish it
  from its sibling.
- Focus on differences, not similarities.

2) Parent payload
- Create a combined parent KnowledgePayload that summarizes and unifies both
  siblings.
- Preserve the same fields and structure as the child payloads.
- Summary and labels should reflect the shared content and broad themes.
- Keep generated title concise.  Do not include terms like "Comprehensive knowledge of", "Integrated overview of..".  Just include topics.

3) Parent refinements
- For each child, generate 3 refinement labels that differentiate the child
  from the parent (more specific versions of parent-level ideas).
  Example:
    parent: "customer support"
    child:  "refund disputes"
- Generate 3 short refinement tags (3–4 words) per child that capture what is
  unique to the child compared to the parent.

Return a single JSON object that matches this structure exactly:

{{
  "sibling_a": {{
    "contrast": ["...", "...", "..."],
    "contrast_tag": ["...", "...", "..."]
  }},
  "sibling_b": {{
    "contrast": ["...", "...", "..."],
    "contrast_tag": ["...", "...", "..."]
  }},
  "parent_payload": {{
    "title": ...,
    "summary": ...,
    "key_topics": [...],
    "entities": [...],
    "facts": [...],
    "intent_purpose": [...],
    "target_audience": [...],
    "document_type": [...],
    "industry": [...],
    "life_domain": [...],
    "provenance": [...]
  }},
  "parent_refinement_for_a": {{
    "parent_refine": ["...", "...", "..."],
    "parent_refine_tag": ["...", "...", "..."]
  }},
  "parent_refinement_for_b": {{
    "parent_refine": ["...", "...", "..."],
    "parent_refine_tag": ["...", "...", "..."]
  }}
}}

Use double quotes and valid JSON only. No markdown, no comments, no extra text.

Sibling A payload:
{payload_a}

Sibling B payload:
{payload_b}

response:
"""
)

In [27]:
#model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)#.with_structured_output(BranchEnrichment)
#branch_chain = branch_template | model
async def process_ready_branches(branches, pay_dict, model_name="gpt-4.1-mini", max_concurrency=8):
    sem = asyncio.Semaphore(max_concurrency)
    model = ChatOpenAI(model=model_name, temperature=0)
    branch_chain = branch_prompt | model.with_structured_output(BranchEnrichment)

    async def run_branch(branch):
        left_id, right_id = branch["node_ids"]
        
        try:
            async with sem:
                result: BranchEnrichment = await branch_chain.ainvoke({
                    "branch_id": str(branch["branch_id"]),
                    "left_node_id": str(left_id),
                    "right_node_id": str(right_id),
                    "left_payload": pay_dict[left_id],
                    "right_payload": pay_dict[right_id],
                })
        except Exception as e:
            return {
                "branch_id": branch["branch_id"],
                "status": "error",
                "error": str(e),
            }

        # parent payload becomes the payload for this branch_id node
        pay_dict[branch["branch_id"]] = result.parent_payload.model_dump()

        # optionally store contrast/refinement somewhere
        sibling_meta = {
            "branch_id": branch["branch_id"],
            "left_node_id": left_id,
            "right_node_id": right_id,
            "sibling_a": result.sibling_a.model_dump(),
            "sibling_b": result.sibling_b.model_dump(),
            "parent_refinement_for_a": result.parent_refinement_for_a.model_dump(),
            "parent_refinement_for_b": result.parent_refinement_for_b.model_dump(),
        }

        return {
            "branch_id": branch["branch_id"],
            "status": "ok",
            "payload": result.parent_payload.model_dump(),
            "enrichment": sibling_meta,
        }

    return await asyncio.gather(*(run_branch(b) for b in branches))

In [28]:
qry = """select branch_id, array_agg(node_id) AS node_ids, sum(case when tz.edge_ix = 1 then 1 else 0 end) as payloads
from graph.v_orchard tz 
where tree_id = '75e5abf9-bf3a-421c-872a-e3a5b81d2eec'
group by branch_id
"""

df_branch = pd.read_sql(qry,engine)
branches = df_branch.to_dict(orient="records")

In [29]:
branch_meta_dict = {}
pay_dict_copy = pay_dict

NameError: name 'pay_dict' is not defined

In [154]:
while True:
    ready = [
        b for b in branches
        if b["branch_id"] not in pay_dict
        and all(node_id in pay_dict for node_id in b["node_ids"])
    ]

    if not ready:
        print("No more ready branches.")
        break

    merged_results = await process_ready_branches(ready, pay_dict)

    new_count = 0
    for r in merged_results:
        if r["status"] == "ok":
            pay_dict[r["branch_id"]] = r["payload"]
            branch_meta_dict[r["branch_id"]] = r["enrichment"]
            new_count += 1

    print(f"Built {new_count} branch payloads this pass.")

    if new_count == 0:
        print("No successful merges this pass; stopping.")
        break

Built 52 branch payloads this pass.
Built 32 branch payloads this pass.
Built 20 branch payloads this pass.
Built 10 branch payloads this pass.
Built 9 branch payloads this pass.
Built 6 branch payloads this pass.
Built 4 branch payloads this pass.
Built 2 branch payloads this pass.
Built 2 branch payloads this pass.
Built 2 branch payloads this pass.
Built 2 branch payloads this pass.
Built 2 branch payloads this pass.
Built 2 branch payloads this pass.
Built 2 branch payloads this pass.
Built 1 branch payloads this pass.
Built 1 branch payloads this pass.
Built 1 branch payloads this pass.
Built 1 branch payloads this pass.
Built 1 branch payloads this pass.
Built 1 branch payloads this pass.
Built 1 branch payloads this pass.
Built 1 branch payloads this pass.
Built 1 branch payloads this pass.
Built 1 branch payloads this pass.
Built 1 branch payloads this pass.
Built 1 branch payloads this pass.
Built 1 branch payloads this pass.
Built 1 branch payloads this pass.
Built 1 branch p

In [146]:
len(ready)

52

In [150]:
ready[0]

{'branch_id': UUID('574987a6-2d79-4372-86b0-d0b5469730a9'),
 'node_ids': [UUID('fa02b4d1-2152-41fb-8939-1c80d178f9f1'),
  UUID('b6cf6148-24a7-470a-82ab-2d575febbcf8')],
 'payloads': 2}

In [3]:
len(pay_dict.keys())

NameError: name 'pay_dict' is not defined

In [159]:
df_pay_dict = pd.DataFrame(branch_meta_dict).T.reset_index()
df_pay_dict.rename(columns={'index':'doc_id'},inplace=True)
df_pay_dict.head()
df_pay_dict.to_csv('df_meta_dict_full.csv',index=False)

In [33]:
df_meta_dict = pd.read_csv('df_meta_dict_full.csv')
df_pay_dict = pd.read_csv('df_pay_dict.csv')
df_pay_dict_full = pd.read_csv('df_pay_dict_full.csv')

df_pay_dict_full.info()

<class 'pandas.DataFrame'>
RangeIndex: 327 entries, 0 to 326
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   doc_id           327 non-null    str  
 1   title            327 non-null    str  
 2   summary          327 non-null    str  
 3   key_topics       327 non-null    str  
 4   entities         327 non-null    str  
 5   facts            327 non-null    str  
 6   intent_purpose   327 non-null    str  
 7   target_audience  327 non-null    str  
 8   document_type    327 non-null    str  
 9   industry         327 non-null    str  
 10  life_domain      327 non-null    str  
 11  provenance       327 non-null    str  
dtypes: str(12)
memory usage: 30.8 KB


In [70]:
df_pay_dict_full.tail()

,doc_id,title,summary,key_topics,entities,facts,intent_purpose,target_audience,document_type,industry,life_domain,provenance
322,b44e1a95-3bd7-48a2-951d-5212cc9efe0a,"Comprehensive Integrated Knowledge on Travel, ...",This synthesized knowledge payload integrates ...,"['Travel and tourism marketing, safety, and re...","['Cape Town Tourism', 'Table Mountain', 'Exped...",['Cape Town International Airport saw over 4.3...,"['Educate visitors about travel destinations, ...","['International and domestic tourists', 'Touri...","['Web content', 'Annual report', 'Safety guide...","['Tourism', 'Hospitality', 'Travel and Leisure...","['Vacation', 'Career', 'Community Engagement',...",['Child node ID: 8cc0eb60-2a25-4bf4-825a-2c0a6...
323,6921fba0-f0c8-41fd-b18e-907780a14ac8,"Integrated Knowledge on Consumer Electronics, ...",This comprehensive knowledge payload synthesiz...,['Aura digital photo frames and troubleshootin...,"['Aura digital photo frame', 'Carver 10"" digit...","['The Carver 10"" frame features a 10.1"" HD lan...","['Inform about product features and benefits',...","['Aura frame users', 'Digital photo frame owne...","['Product description', 'Sales advertisement',...","['Consumer Electronics', 'Retail', 'Tourism', ...","['Home', 'Gifting', 'Vacation', 'Career', 'Com...",['Child node ID: 42a13de7-4ad8-470b-8ab9-2a3a0...
324,719d6f4f-7517-4298-aac7-ee902d47cd31,Comprehensive Knowledge on Consumer Technology...,This synthesized knowledge payload integrates ...,['Consumer electronics and Aura digital photo ...,"['Aura digital photo frame', 'Carver 10"" digit...","['The Carver 10"" frame features a 10.1"" HD lan...","['Inform about product features and benefits',...","['Aura frame users', 'Digital photo frame owne...","['Product description', 'Sales advertisement',...","['Consumer Electronics', 'Retail', 'Tourism', ...","['Home', 'Gifting', 'Vacation', 'Career', 'Com...",['Child node ID: 6921fba0-f0c8-41fd-b18e-90778...
325,aff3d9f8-84ff-45e3-b6d4-5c6ee184e368,Integrated Knowledge on Data Insertion Techniq...,This synthesized knowledge payload combines fo...,"['pandas DataFrame insertion into PostgreSQL',...","['pandas', 'SQLAlchemy', 'PostgreSQL', 'DataFr...",['DataFrame.to_sql can insert data into Postgr...,['Educate on inserting pandas DataFrame into P...,"['Data scientists', 'Python developers', 'Data...","['Tutorial', 'Code example', 'Product descript...","['Information Technology', 'Consumer Electroni...","['Career', 'Education', 'Home', 'Gifting', 'Va...",['Child node ID: 6e561c16-4b55-42a8-a53a-f10a8...
326,503a5c41-cec1-4afb-bdca-884e53d955a3,Comprehensive Knowledge Synthesis: Culinary Ar...,This integrated knowledge payload combines det...,['Cherry pie and cornbread recipes and baking ...,"['Butter pie crust', 'Fresh cherries', 'Granul...",['A 9-inch double-crust homemade butter pie cr...,['Inform about detailed baking and cooking tec...,"['Home bakers', 'Baking enthusiasts', 'Cooking...","['Recipe', 'Instructional article', 'Cooking t...","['Food & Beverage', 'Culinary Arts', 'Baking',...","['Home', 'Cooking', 'Food Preparation', 'Educa...",['Child node ID: 97e878fe-ba4c-4d17-bc1b-425f4...


In [68]:
meta_list[10]

{'doc_id': '4bc8cbac-8b1c-4033-bae3-212cb3c1ece0',
 'branch_id': '4bc8cbac-8b1c-4033-bae3-212cb3c1ece0',
 'left_node_id': '87e8c434-c99c-4a8e-b494-b8ccc1f132e2',
 'right_node_id': 'eb5cf10b-de6d-4c8b-8d21-ca3aa3314514',
 'sibling_a': "{'contrast': ['Focuses on converting screen recordings into step-by-step SOPs automatically, emphasizing process documentation and operational efficiency.', 'Supports over 140 languages and integrates with popular video platforms like Loom, Zoom, and Teams.', 'Targets operations, quality management, training, manufacturing, IT, healthcare, customer service, sales, HR, and compliance teams.', 'Highlights rapid SOP generation, multilingual support, and ease of updating documentation via new recordings.'], 'contrast_tag': ['Process Documentation', 'Video to SOP', 'Operational Efficiency', 'Multilingual Support', 'Integration with Video Platforms', 'Targeted Industries', 'Rapid SOP Generation']}",
 'sibling_b': "{'contrast': ['Offers AI video creation softwar

In [74]:
import ast

full_list = df_pay_dict_full.to_dict(orient='records')
full_dict = {d['doc_id']:d for d in full_list}

meta_list = df_meta_dict.to_dict(orient='records')
meta_dict = {d['doc_id']:d for d in full_list}

node_dict = {
    node_id: {
        **ast.literal_eval(sibling),
        **ast.literal_eval(parent)
    }
    for item in meta_list
    for node_id, sibling, parent in [
        (item["left_node_id"], item["sibling_a"], item["parent_refinement_for_a"]),
        (item["right_node_id"], item["sibling_b"], item["parent_refinement_for_b"]),
    ]
}

merged = {
    k: {**full_dict.get(k, {}), **node_dict.get(k, {})}
    for k in set(full_dict) | set(node_dict)
}

In [75]:
merged['87e8c434-c99c-4a8e-b494-b8ccc1f132e2']

{'doc_id': '87e8c434-c99c-4a8e-b494-b8ccc1f132e2',
 'title': 'Video to SOP',
 'summary': 'Video to SOP is an AI-powered tool that converts screen recordings into standardized step-by-step standard operating procedures (SOPs) automatically. It analyzes videos to extract key steps, screenshots, timestamps, and instructions, enabling organizations to generate professional documentation quickly and consistently. The tool supports over 140 languages, works with various video sources, and allows easy updates by recording new videos instead of rewriting documents. It is used across industries for onboarding, compliance, training, and process documentation, saving significant time compared to manual writing.',
 'key_topics': "['AI-powered video analysis', 'Standard operating procedure (SOP) generation', 'Screen recording conversion', 'Automatic screenshot capture and annotation', 'Multilingual support', 'Process documentation', 'Training and onboarding', 'Compliance and audit preparation', 'Ex

In [60]:
df_full_json = pd.DataFrame({'doc_id':list(merged.keys()),"payload":list(merged.values())})
from psycopg2.extras import Json
import uuid

#df_full_json["payload"] = df_full_json["payload"].apply(Json)
#df_full_json["doc_id"] = df_full_json["doc_id"].apply(uuid.UUID)

df_full_json.head()

,doc_id,payload
0,b0493307-1e7a-4fbf-92fc-7639c676b5a1,{'doc_id': 'b0493307-1e7a-4fbf-92fc-7639c676b5...
1,51c47e1a-f80f-4160-a6ff-939cc56d4266,{'doc_id': '51c47e1a-f80f-4160-a6ff-939cc56d42...
2,d0a9221f-b8db-495c-b6a8-ed0cefdc906b,{'doc_id': 'd0a9221f-b8db-495c-b6a8-ed0cefdc90...
3,14ade5c6-f0cb-4b6c-b8f9-7c08b64fe08d,{'doc_id': '14ade5c6-f0cb-4b6c-b8f9-7c08b64fe0...
4,3204a0cf-8977-410d-b695-77b08732ecdd,{'doc_id': '3204a0cf-8977-410d-b695-77b08732ec...


In [50]:
from sqlalchemy.dialects.postgresql import JSONB, UUID

df_full_json.to_sql(
    "node_enrichment",          # table name
    con=engine,
    schema="graph",
    if_exists="append",  # create if not exists, append if exists
    index=False,         # don’t write the index as a column
    dtype={
        "doc_id": UUID,
        "payload": JSONB,
    },
    chunksize=1000,
    method="multi",
)


327

In [57]:
from sqlalchemy.dialects.postgresql import insert
from sqlalchemy import Table, MetaData

metadata = MetaData()

node_enrichment = Table(
    "node_enrichment",
    metadata,
    schema="graph",
    autoload_with=engine,
)

def upsert_df(df, table, engine, conflict_cols, update_cols, chunksize=1000):
    records = df.to_dict(orient="records")

    with engine.begin() as conn:
        for i in range(0, len(records), chunksize):
            chunk = records[i:i + chunksize]

            stmt = insert(table).values(chunk)

            upsert_stmt = stmt.on_conflict_do_update(
                index_elements=conflict_cols,
                set_={
                    col: getattr(stmt.excluded, col)
                    for col in update_cols
                },
            )

            conn.execute(upsert_stmt)

In [58]:
upsert_df(
    df_full_json,
    node_enrichment,
    engine,
    conflict_cols=["doc_id"],
    update_cols=["payload"],
)

In [61]:
import ast
import json
import uuid

def normalize_payload(p):
    if isinstance(p, str):
        p = json.loads(p)

    out = {}
    for k, v in p.items():
        if isinstance(v, str) and v.startswith("[") and v.endswith("]"):
            try:
                v = ast.literal_eval(v)
            except Exception:
                pass
        out[k] = v

    return out

df_insert = df_full_json.copy()
df_insert["doc_id"] = df_insert["doc_id"].apply(lambda x: str(uuid.UUID(str(x))))
df_insert["payload"] = df_insert["payload"].apply(normalize_payload)

In [62]:
import io
import csv
from sqlalchemy import text

def copy_upsert_node_enrichment(df, engine):
    df = df[["doc_id", "payload"]].copy()
    df["doc_id"] = df["doc_id"].apply(lambda x: str(uuid.UUID(str(x))))
    df["payload"] = df["payload"].apply(lambda x: json.dumps(normalize_payload(x)))

    buf = io.StringIO()
    df.to_csv(buf, index=False, header=False, quoting=csv.QUOTE_MINIMAL)
    buf.seek(0)

    copy_sql = """
        COPY tmp_node_enrichment (doc_id, payload)
        FROM STDIN WITH (FORMAT csv)
    """

    upsert_sql = """
        INSERT INTO graph.node_enrichment (doc_id, payload)
        SELECT doc_id, payload
        FROM tmp_node_enrichment
        ON CONFLICT (doc_id)
        DO UPDATE SET
            payload = EXCLUDED.payload;
    """

    with engine.begin() as conn:
        conn.execute(text("""
            CREATE TEMP TABLE tmp_node_enrichment (
                doc_id uuid,
                payload jsonb
            ) ON COMMIT DROP;
        """))

        raw = conn.connection.driver_connection
        with raw.cursor() as cur:
            cur.copy_expert(copy_sql, buf)

        conn.execute(text(upsert_sql))

In [63]:
copy_upsert_node_enrichment(df_insert, engine)

In [169]:
qry = """SELECT *
FROM graph.get_compressed_subtree(
  (SELECT array_agg(DISTINCT orchard_id) FROM graph.v_orchard tz 
where tree_id = '75e5abf9-bf3a-421c-872a-e3a5b81d2eec'
and edge_ix =1))
"""
df_tree = pd.read_sql(qry,engine)
df_tree.head()

,tree_id,branch_id,branch_ix,l_node_ix,r_node_ix,l_node_id,r_node_id,l_edge_ix,r_edge_ix,doc_count,coph_distance
0,75e5abf9-bf3a-421c-872a-e3a5b81d2eec,25a5d008-6902-497d-a528-7bbb9aa67b9f,165,2,1,b61e0cb4-6fb7-4a3e-a91c-d91709496299,33aff4d2-9d89-4c9b-878d-e754d0d770a0,1,1,2,0.086124
1,75e5abf9-bf3a-421c-872a-e3a5b81d2eec,dc1a0d28-bb29-4b38-8626-21a023bffa67,166,4,3,2d91e5e8-548a-4fb4-841d-93b50c94dc28,cc01d4b1-3430-4703-a8f2-d298ca660078,1,1,2,0.097535
2,75e5abf9-bf3a-421c-872a-e3a5b81d2eec,ecee6fae-8a38-4728-98c2-67c9041a218e,167,6,5,d4008893-86cc-4ddd-b049-707548cf1363,56729489-b340-4abc-9150-b370d8d430a8,1,1,2,0.110823
3,75e5abf9-bf3a-421c-872a-e3a5b81d2eec,78b9ac59-893a-4b03-9d4e-3cfd76e10c5d,168,7,8,17de6ee5-1a1e-498d-bc9e-4f9d58cb6270,0f49ae76-b014-4b22-bf26-d227a4360330,1,1,2,0.111063
4,75e5abf9-bf3a-421c-872a-e3a5b81d2eec,b07758e1-35b9-4c1c-a721-056d6a9deb20,169,165,9,25a5d008-6902-497d-a528-7bbb9aa67b9f,4d6b18e9-b18a-48d0-a6f7-9e7796d1d1d7,2,1,3,0.116608


In [201]:
#df_tree['l_node_ix']=df_tree['l_node_ix']-1
#df_tree['r_node_ix']=df_tree['r_node_ix']-1

Z = df_tree[['l_node_ix', 'r_node_ix', 'coph_distance', 'doc_count']].to_numpy()
Z.shape

(163, 4)

In [172]:
from scipy.cluster.hierarchy import dendrogram


In [193]:
df_tree[['l_node_ix', 'r_node_ix', 'coph_distance', 'doc_count']].head(10)

,l_node_ix,r_node_ix,coph_distance,doc_count
0,2,1,0.086124,2
1,4,3,0.097535,2
2,6,5,0.110823,2
3,7,8,0.111063,2
4,165,9,0.116608,3
5,10,11,0.143029,2
6,13,12,0.143978,2
7,14,15,0.162355,2
8,16,17,0.186522,2
9,19,18,0.186629,2


In [202]:
n=Z.shape[0]
np.any(np.max(Z[:, :2], axis=1) >= np.arange(n + 1, 2 * n + 1, dtype=Z.dtype))
np.max(Z[:, :2])


np.float64(325.0)

In [204]:
from scipy.cluster.hierarchy import fcluster

n_clusters = 5
labels = fcluster(Z, n_clusters, criterion="maxclust")


In [205]:
labels

array([4, 4, 4, 4, 4, 4, 2, 2, 4, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
       4, 4, 4, 4, 4, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
       4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
       4, 4, 4, 4, 4, 4, 4, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
       4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 4, 4,
       4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 4, 4, 4, 4, 4, 4, 4,
       4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
       4, 4, 4, 2, 4, 4, 4, 4, 4, 5], dtype=int32)